<a href="https://colab.research.google.com/github/ShahadAbdullahDS/SARF-banking-nlp/blob/main/03_notebooks/02_svm_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SARF — SVM Baseline

**Run ID:** `svm_msa_validation_v1`  
**Model:** TF-IDF + LinearSVC  
**Seed:** 42  
**Training data:** MSA train only  
**Evaluation data:** MSA validation only  
**Primary metric:** Macro-F1  

## Protocol restrictions

- Saudi test was not accessed.
- MSA test was not used.
- Synthetic corpora were not used.
- The smoke run is separate from the official baseline result.
- Model selection and evaluation use MSA validation only.


In [ ]:
from google.colab import drive
from pathlib import Path

import sys
import json
import shutil
import joblib
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix

drive.mount("/content/drive")

DATA_DIR = Path(
    "/content/drive/MyDrive/"
    "SARF_BANKING_NLP_PROJECT/02_processed_data"
)

RUN_DIR = Path(
    "/content/drive/MyDrive/"
    "SARF_BANKING_NLP_PROJECT/05_runs/"
    "baselines/svm_msa_validation_v1"
)

RUN_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Drive connected")
print("Data directory:", DATA_DIR)
print("Run directory:", RUN_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive connected
Data directory: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_processed_data
Run directory: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/05_runs/baselines/svm_msa_validation_v1


In [ ]:
from google.colab import userdata
import subprocess
import os
import tempfile

PROJECT_DIR = Path("/content/SARF-banking-project")

if not (PROJECT_DIR / ".git").exists():
    token = userdata.get("GITHUB_TOKEN")

    askpass_code = """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""

    with tempfile.NamedTemporaryFile(
        mode="w",
        delete=False
    ) as file:
        file.write(askpass_code)
        askpass_path = file.name

    os.chmod(askpass_path, 0o700)

    git_env = os.environ.copy()
    git_env["GIT_ASKPASS"] = askpass_path
    git_env["GITHUB_TOKEN"] = token
    git_env["GIT_TERMINAL_PROMPT"] = "0"

    result = subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/ShahadAbdullahDS/SARF-banking-nlp.git",
            str(PROJECT_DIR)
        ],
        env=git_env,
        capture_output=True,
        text=True
    )

    os.remove(askpass_path)
    del token
    git_env.pop("GITHUB_TOKEN", None)

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

print("✅ Project code is available")

sys.path.insert(0, str(PROJECT_DIR / "04_src"))

from baseline_svm import run_svm_baseline

✅ Project code is available


In [ ]:
TRAIN_PATH = DATA_DIR / "msa_train_v1.csv"
VAL_PATH = DATA_DIR / "msa_val_v1.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

required_columns = {"label", "text"}

assert required_columns.issubset(train_df.columns)
assert required_columns.issubset(val_df.columns)
assert len(train_df) == 10732
assert len(val_df) == 1229
assert train_df["label"].nunique() == 77
assert val_df["label"].nunique() == 77
assert train_df[["label", "text"]].isna().sum().sum() == 0
assert val_df[["label", "text"]].isna().sum().sum() == 0

unknown_labels = (
    set(val_df["label"]) - set(train_df["label"])
)
assert len(unknown_labels) == 0

assert not train_df["text"].astype(str).str.strip().eq("").any()
assert not val_df["text"].astype(str).str.strip().eq("").any()

print("✅ Data validation passed")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Labels:", train_df["label"].nunique())
print("Unknown validation labels:", len(unknown_labels))

display(train_df.head(3))

✅ Data validation passed
Train: (10732, 2)
Validation: (1229, 2)
Labels: 77
Unknown validation labels: 0


,label,text
0,وصول البطاقة,ما زلت أنتظر بطاقتي؟
1,وصول البطاقة,ماذا أفعل إذا لم تصل بطاقتي بعد أسبوعين؟
2,وصول البطاقة,انا أنتظر منذ أكثر من أسبوع. هل ما زالت البطاق...


In [ ]:
SMOKE_DIR = RUN_DIR / "smoke"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

# عينة ثابتة من كل label
smoke_train = pd.concat(
    [
        group.sample(
            n=min(20, len(group)),
            random_state=42
        )
        for _, group in train_df.groupby("label")
    ],
    ignore_index=True
)

smoke_val = pd.concat(
    [
        group.sample(
            n=min(5, len(group)),
            random_state=42
        )
        for _, group in val_df.groupby("label")
    ],
    ignore_index=True
)

smoke_train_path = SMOKE_DIR / "smoke_train.csv"
smoke_val_path = SMOKE_DIR / "smoke_val.csv"

smoke_train.to_csv(smoke_train_path, index=False)
smoke_val.to_csv(smoke_val_path, index=False)

run_svm_baseline(
    train_csv=smoke_train_path,
    evaluation_csv=smoke_val_path,
    output_dir=SMOKE_DIR,
    run_id="svm_msa_smoke_seed_42",
    evaluation_split="msa_val",
    seed=42
)

with open(
    SMOKE_DIR / "metrics.json",
    encoding="utf-8"
) as file:
    smoke_metrics = json.load(file)

smoke_predictions = pd.read_csv(
    SMOKE_DIR / "predictions.csv"
)

assert smoke_train["label"].nunique() == 77
assert smoke_val["label"].nunique() == 77
assert len(smoke_predictions) == len(smoke_val)
assert smoke_predictions["y_pred"].nunique() > 1

print("✅ Smoke run passed")
print("Train rows:", len(smoke_train))
print("Validation rows:", len(smoke_val))
print("Labels:", smoke_train["label"].nunique())
print(
    "Unique predicted labels:",
    smoke_predictions["y_pred"].nunique()
)
print(
    "Smoke Macro-F1:",
    round(smoke_metrics["macro_f1"], 6)
)

✅ Smoke run passed
Train rows: 1540
Validation rows: 384
Labels: 77
Unique predicted labels: 77
Smoke Macro-F1: 0.625946


In [ ]:
official_paths = run_svm_baseline(
    train_csv=TRAIN_PATH,
    evaluation_csv=VAL_PATH,
    output_dir=RUN_DIR,
    run_id="svm_msa_validation_v1",
    evaluation_split="msa_val",
    seed=42
)

with open(
    RUN_DIR / "metrics.json",
    encoding="utf-8"
) as file:
    official_metrics = json.load(file)

official_predictions = pd.read_csv(
    RUN_DIR / "predictions.csv"
)

assert official_metrics["n_examples"] == 1229
assert official_metrics["n_labels_in_protocol"] == 77
assert len(official_predictions) == 1229
assert official_predictions["y_pred"].nunique() > 1

print("✅ Official SVM run completed")
print(
    "Macro-F1:",
    round(official_metrics["macro_f1"], 6)
)
print(
    "Weighted F1:",
    round(official_metrics["weighted_f1"], 6)
)
print(
    "Accuracy:",
    round(official_metrics["accuracy"], 6)
)
print(
    "Prediction rows:",
    len(official_predictions)
)

✅ Official SVM run completed
Macro-F1: 0.86721
Weighted F1: 0.873176
Accuracy: 0.873881
Prediction rows: 1229


In [ ]:
# توحيد أسماء الملفات المطلوبة
file_copies = {
    "metrics.json": "validation_metrics.json",
    "predictions.csv": "validation_predictions.csv",
    "per_intent_metrics.csv": "per_class_metrics.csv",
    "model.joblib": "svm_tfidf.joblib",
    "run_config.json": "config.json",
}

for source_name, target_name in file_copies.items():
    shutil.copy2(
        RUN_DIR / source_name,
        RUN_DIR / target_name
    )

# تحميل الإعدادات والتنبؤات
with open(
    RUN_DIR / "config.json",
    encoding="utf-8"
) as file:
    config = json.load(file)

predictions = pd.read_csv(
    RUN_DIR / "validation_predictions.csv"
)

label_order = config["label_order"]

# Confusion matrix
matrix = confusion_matrix(
    predictions["y_true"],
    predictions["y_pred"],
    labels=label_order
)

confusion_df = pd.DataFrame(
    matrix,
    index=label_order,
    columns=label_order
)
confusion_df.index.name = "true_label"

confusion_df.to_csv(
    RUN_DIR / "confusion_matrix.csv",
    encoding="utf-8-sig"
)

# Reload test
reloaded_model = joblib.load(
    RUN_DIR / "svm_tfidf.joblib"
)

reload_texts = val_df["text"].iloc[:3]
reload_predictions = reloaded_model.predict(reload_texts)

reload_test = pd.DataFrame({
    "text": reload_texts,
    "true_label": val_df["label"].iloc[:3],
    "predicted_label": reload_predictions
})

reload_test.to_csv(
    RUN_DIR / "reload_test.csv",
    index=False,
    encoding="utf-8-sig"
)

assert len(reload_predictions) == 3

# Evaluation summary
evaluation_summary = {
    "run_id": "svm_msa_validation_v1",
    "owner": "A",
    "model_family": "TF-IDF + LinearSVC",
    "condition": "baseline_svm",
    "run_type": "official_baseline",
    "seed": 42,
    "train_split": "msa_train_v1",
    "validation_split": "msa_val_v1",
    "train_rows": 10732,
    "validation_rows": 1229,
    "num_labels": 77,
    "metrics": {
        "macro_f1": official_metrics["macro_f1"],
        "weighted_f1": official_metrics["weighted_f1"],
        "accuracy": official_metrics["accuracy"]
    },
    "prediction_file": "validation_predictions.csv",
    "per_class_file": "per_class_metrics.csv",
    "confusion_matrix_file": "confusion_matrix.csv",
    "model_file": "svm_tfidf.joblib",
    "reload_test_passed": True,
    "saudi_test_accessed": False,
    "data_provenance": (
        "MSA splits reconstructed from the public ArBanking77 "
        "MSA_PAL files using the project-approved row boundaries; "
        "official Drive hashes pending verification."
    ),
    "status": "completed_pending_data_hash_verification"
}

with open(
    RUN_DIR / "evaluation_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        evaluation_summary,
        file,
        ensure_ascii=False,
        indent=2
    )

# Run log
run_log = f"""# SVM Run Log

## Run information

- Run ID: `svm_msa_validation_v1`
- Model: TF-IDF + LinearSVC
- Seed: 42
- Train rows: 10,732
- Validation rows: 1,229
- Labels: 77
- Unknown validation labels: 0
- Saudi test accessed: false
- Synthetic data used: false
- Reload test passed: true

## Configuration

- Analyzer: word
- N-grams: (1, 2)
- Minimum document frequency: 1
- Maximum features: 100,000
- Sublinear TF: true
- Lowercase: false
- Class weight: balanced

## Validation metrics

- Macro-F1: {official_metrics["macro_f1"]:.6f}
- Weighted F1: {official_metrics["weighted_f1"]:.6f}
- Accuracy: {official_metrics["accuracy"]:.6f}

## Data provenance

The MSA files were reconstructed from the public ArBanking77
MSA_PAL files using the project-approved counts of 10,732 training
rows and 1,229 validation rows. Their contents or hashes must be
compared with the official Drive copies before final approval.

## Smoke issue

One validation label had fewer than five examples. The smoke sampler
was changed to use up to five rows per label. The smoke run still
covered all 77 labels and passed.
"""

with open(
    RUN_DIR / "run_log.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(run_log)

# فحوص نهائية
assert len(predictions) == 1229
assert len(
    pd.read_csv(RUN_DIR / "per_class_metrics.csv")
) == 77
assert matrix.shape == (77, 77)

required_files = [
    "config.json",
    "run_log.md",
    "validation_metrics.json",
    "evaluation_summary.json",
    "validation_predictions.csv",
    "per_class_metrics.csv",
    "confusion_matrix.csv",
    "svm_tfidf.joblib"
]

for filename in required_files:
    assert (RUN_DIR / filename).exists()

print("✅ All required artifacts created")
print("✅ Reload test passed")
display(reload_test)

print("\nRequired files:")
for filename in required_files:
    print("✅", filename)

✅ All required artifacts created
✅ Reload test passed


,text,true_label,predicted_label
0,هل يمكنك تتبع بطاقتي من أجلي؟,وصول البطاقة,وصول البطاقة
1,هل سأتمكن من تتبع البطاقة التي تم إرسالها إلي؟,وصول البطاقة,وصول البطاقة
2,بطاقتي ليست هنا بعد.,وصول البطاقة,وصول البطاقة



Required files:
✅ config.json
✅ run_log.md
✅ validation_metrics.json
✅ evaluation_summary.json
✅ validation_predictions.csv
✅ per_class_metrics.csv
✅ confusion_matrix.csv
✅ svm_tfidf.joblib


In [ ]:
import json

# تحديث evaluation_summary.json
summary_path = RUN_DIR / "evaluation_summary.json"

with open(summary_path, encoding="utf-8") as file:
    summary = json.load(file)

summary["data_provenance"] = (
    "Official approved MSA train and validation "
    "splits loaded directly from the team Drive."
)

summary["status"] = "completed"

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2
    )

# تصحيح قسم Data provenance في run_log.md
log_path = RUN_DIR / "run_log.md"
log_text = log_path.read_text(encoding="utf-8")

start = log_text.find("## Data provenance")
end = log_text.find("## Smoke issue")

official_provenance = """## Data provenance

The official approved MSA train and validation files were loaded
directly from the team Drive.

"""

if start != -1 and end != -1:
    log_text = (
        log_text[:start]
        + official_provenance
        + log_text[end:]
    )

log_path.write_text(
    log_text,
    encoding="utf-8"
)

print("✅ SVM status updated to completed")
print("Data source: official team Drive")
print("Macro-F1:", round(summary["metrics"]["macro_f1"], 6))

✅ SVM status updated to completed
Data source: official team Drive
Macro-F1: 0.86721
